# 1. Define constants

In [12]:
VIBLO_ARTICLE_URL = 'https://viblo.asia/newest?page={page}' # Default page = 1

# leave page empty
url = VIBLO_ARTICLE_URL.format(page='')

# or fill it later
url = VIBLO_ARTICLE_URL.format(page=2)

In [ ]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag
from typing import Optional, List, Dict
import urllib.parse
import json
import datetime

def get_avaiable_articles(page: int = 1, verbose: bool = True, auto_save: str = '') -> List[Dict]:
    if page < 1:
        print(f'[WARNING]: Page number {page} is invalid. This will be ignored by the API.')
    # Fetch articles from the API
    if verbose:
        print(f'Fetching articles from page {page}...')
    api_url = VIBLO_ARTICLE_URL.format(page=page)

    # Get the API response
    response = requests.get(api_url)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch articles: {response.status_code}")

    # Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')

    # Get all article wrapper elements, contains link to the article and tags
    """
    Example HTML structure:
    <div class="post-title--inline">
        <h3 class="word-break mr-05">
            <a href="/p/homelab-19-cai-dat-apache-guacamole-pPLkNN3ZJRZ" class="link">[Homelab] #19 Cài đặt Apache Guacamole</a>
        </h3>
        <div class="tags d-flex flex-wrap" data-v-4365a2a0>
            <a href="/tags/homelab" class="el-tag tag el-tag--info el-tag--mini" data-v-22b6e812 data-v-4365a2a0>homelab</a>
        </div>
    </div>
    """
    ARTICLE_WRAPPER_CLASS = 'post-title--inline'
    ARTICLE_LINK_CLASS    = 'link'
    ARTICLE_TAG_CLASS     = 'el-tag tag el-tag--info el-tag--mini'

    article_elements = soup.find_all(class_=ARTICLE_WRAPPER_CLASS)

    if verbose:
        print(f'Found {len(article_elements)} articles on page {page}.')

    crawled_articles = []

    for article in article_elements:
        # Get class=link elements inside article_elements
        link_element: Optional[Tag] = article.find(class_=ARTICLE_LINK_CLASS)

        # Get all tags into a list from class='el-tag tag el-tag--info el-tag--mini'
        tags = []
        tag_elements = article.find_all(class_=ARTICLE_TAG_CLASS)
        for tag_element in tag_elements:
            tags.append(tag_element.get_text(strip=True))

        # If there is no link element, skip this article to avoid None attribute/subscript errors
        if not link_element:
            # Optionally log or collect placeholders instead of skipping
            continue

        # Use safe accessors: get_text and .get('href') to avoid 'None' / subscript issues
        name = link_element.get_text(strip=True)

        # BeautifulSoup attribute access can return different types (e.g., list-like).
        # Normalize href to a plain string before passing to urljoin to satisfy type checkers.
        href_attr = link_element.get('href')
        if isinstance(href_attr, list):
            href = href_attr[0] if href_attr else ''
        else:
            href = href_attr or ''

        full_link = urllib.parse.urljoin('https://viblo.asia', str(href))

        crawled_articles.append({
            'name': name,
            'link': full_link,
            'tags': tags
        })

    if auto_save != '':
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        with open(f'{auto_save}/crawled_articles_page_{page}_{timestamp}.json', 'w', encoding='utf-8') as f:
            json.dump(crawled_articles, f, ensure_ascii=False, indent=4)
        print(f'Auto-saved crawled articles to {auto_save}/crawled_articles_page_{page}_{timestamp}.json')

    return crawled_articles


get_avaiable_articles(page=1, auto_save='../data')

Fetching articles from page 15...
Found 21 articles on page 15.
Auto-saved crawled articles to ../data/crawled_articles_page_15_20251024_155714.json


[{'name': 'VIBLO MOBILE APP CHÍNH THỨC RA MẮT – TRẢI NGHIỆM NGAY VÀ THAM GIA MINIGAME HẤP DẪN! 📲',
  'link': 'https://viblo.asia/announcements/viblo-mobile-app-chinh-thuc-ra-mat-trai-nghiem-ngay-va-tham-gia-minigame-hap-dan-GyZJZo7GLjm',
  'tags': []},
 {'name': 'Tại sao môi trường phát triển trên Windows lại khó cấu hình đến vậy, và vì sao vẫn có quá nhiều người dùng?',
  'link': 'https://viblo.asia/p/tai-sao-moi-truong-phat-trien-tren-windows-lai-kho-cau-hinh-den-vay-va-vi-sao-van-co-qua-nhieu-nguoi-dung-gdJzvbxEJz5',
  'tags': ['python windows', 'windows', 'Python', 'servbay']},
 {'name': 'Hướng dẫn sử dụng Git',
  'link': 'https://viblo.asia/p/huong-dan-su-dung-git-kNLr3dmwVgA',
  'tags': ['GitHub']},
 {'name': 'Sự Thật Trần Trụi Về Vector Search: Tại Sao Model Lớn Nhất Cũng Sẽ Đầu Hàng',
  'link': 'https://viblo.asia/p/su-that-tran-trui-ve-vector-search-tai-sao-model-lon-nhat-cung-se-dau-hang-bNVQGW1pJvR',
  'tags': ['vector embedding', 'RAG', 'retrieval', 'AI', 'LLM']},
 {'name':

# 2. GATHER AND TRANSFORM

In [ ]:
import pandas as pd
import os
import json

ARTICLE_URL_DIR = r'D:\LLM\techblog-summarize\data\raw_article_urls'

def gather_article_urls(article_url_dir: str = ARTICLE_URL_DIR, save_dir: str = '', verbose: bool = False) -> pd.DataFrame:
    """
    Gather article URLs from JSON files in the specified directory, remove duplicates, and save to a CSV file.
    Args:
        article_url_dir (str): Directory containing JSON files with article URLs.
        save_dir (str): Path and name to save the gathered CSV file. If empty it won't be saved.
        verbose (bool): If True, prints progress messages.
    """

    # This function will gather article URLs and remove duplicates and save as csv file
    all_files = [f for f in os.listdir(article_url_dir) if f.endswith('.json')]

    if verbose:
        print(f'Found {len(all_files)} JSON files in {article_url_dir}.')

    all_articles = []
    for file in all_files:
        file_path = os.path.join(article_url_dir, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            articles = json.load(f)
            all_articles.extend(articles)

    # Remove duplicates
    all_articles = list({article['link']: article for article in all_articles}.values())

    if verbose:
        print(f'Found {len(all_articles)} unique articles.')

    # Save to CSV
    df = pd.DataFrame(all_articles)

    if save_dir != '':
        df.to_csv(save_dir, index=False)

        if verbose:
            print(f'Successfully gathered articles to {save_dir}...')

    return df

gather_article_urls(save_dir=r'D:\LLM\techblog-summarize\data\gathered_urls\DATA-gathered_article_urls.csv', verbose=True)

Found 15 JSON files in D:\LLM\techblog-summarize\data\raw_article_urls.
Found 299 unique articles.
Successfully gathered articles to D:\LLM\techblog-summarize\data\gathered_urls\DATA-gathered_article_urls.csv...


,name,link,tags
0,VIBLO MOBILE APP CHÍNH THỨC RA MẮT – TRẢI NGHI...,https://viblo.asia/announcements/viblo-mobile-...,[]
1,Padding trong struct,https://viblo.asia/p/padding-trong-struct-kY4g...,[C/Cpp]
2,"Saudi Arabia AI: Understanding Its Types, and ...",https://viblo.asia/p/saudi-arabia-ai-understan...,"[Artificial Intelligence, Saudi Arabia Ai]"
3,Hướng dẫn tạo TLS Cetificiate Fullchain Self-s...,https://viblo.asia/p/huong-dan-tao-tls-cetific...,[Cài đặt SSL]
4,Tạo component React Markdown Preview và Publis...,https://viblo.asia/p/tao-component-react-markd...,"[Markdown, npm]"
...,...,...,...
294,NoSQL là gì? Bước Ngoặt Của Dữ Liệu Lớn Và Ứng...,https://viblo.asia/p/nosql-la-gi-buoc-ngoat-cu...,"[NoSQL, Database, MongoDB, SQL, JSON]"
295,[Playwright Interview question #24]: Các loại ...,https://viblo.asia/p/playwright-interview-ques...,"[Playwright, Playwright Việt Nam, wait]"
296,"Xây dựng ứng dụng bằng NestJS, k8s, ArgoCD, Te...",https://viblo.asia/p/xay-dung-ung-dung-bang-ne...,"[ArgoCD, K8s, terraform]"
297,Vì sao Twilio Segment nói lời chia tay Microse...,https://viblo.asia/p/vi-sao-twilio-segment-noi...,"[microservices, System Design]"
